# DEMO TRỰC QUAN: CƠ CHẾ MERGE INTO TRONG MICROSOFT FABRIC LAKEHOUSE

Notebook này sẽ giúp bạn kiểm chứng thực tế:
1. **Khởi tạo** bảng ban đầu với 3 khách hàng (Version 0).
2. **Mô phỏng** đợt CDC gồm cả 3 thao tác: `INSERT (i)`, `UPDATE (u)`, `DELETE (d)`.
3. **Chạy lệnh MERGE INTO** và quan sát cách Delta Lake xử lý.
4. **Kiểm chứng Transaction Log** xem Delta Lake ghi nhận version và operation metrics ra sao.
5. **Dùng Time Travel** quay ngược thời gian để chứng minh dữ liệu cũ không hề bị mất!

In [ ]:
# BƯỚC 1: KHỞI TẠO BẢNG DỮ LIỆU BAN ĐẦU (VERSION 0)
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime

# Tạo dữ liệu giả lập 3 khách hàng ban đầu
initial_data = [
    ('CUST_001', 'Nguyen Van A', 'SILVER', 'Ha Noi', datetime(2026, 1, 1, 8, 0)),
    ('CUST_002', 'Tran Thi B', 'STANDARD', 'Da Nang', datetime(2026, 1, 1, 8, 0)),
    ('CUST_003', 'Le Van C', 'GOLD', 'TP HCM', datetime(2026, 1, 1, 8, 0))
]

schema = ['customer_id', 'customer_name', 'customer_type', 'address', 'updated_at']
df_init = spark.createDataFrame(initial_data, schema)

# Ghi vào bảng Delta Lake demo
table_name = 'lh_silver.dbo.demo_customers'
df_init.write.format('delta').mode('overwrite').saveAsTable(table_name)

print('--- DỮ LIỆU BAN ĐẦU (VERSION 0) ---')
spark.table(table_name).show()

In [ ]:
# BƯỚC 2: MÔ PHỎNG LUỒNG CDC TỪ KAFKA (CHỨA CẢ I, U, D)
# Giả sử sau 2 tiếng, hệ thống phát sinh các sự kiện sau:
# - CUST_001: Update lên hạng GOLD (cờ u)
# - CUST_002: Hủy tài khoản (cờ d)
# - CUST_003: Giữ nguyên (không có trong CDC)
# - CUST_004: Khách hàng mới tinh đăng ký tài khoản (cờ i)

cdc_data = [
    ('CUST_001', 'Nguyen Van A', 'GOLD', 'Ha Noi', datetime(2026, 1, 1, 10, 0), 'u'),
    ('CUST_002', 'Tran Thi B', 'STANDARD', 'Da Nang', datetime(2026, 1, 1, 10, 0), 'd'),
    ('CUST_004', 'Pham Van D', 'SILVER', 'Hai Phong', datetime(2026, 1, 1, 10, 0), 'i')
]

cdc_schema = ['customer_id', 'customer_name', 'customer_type', 'address', 'updated_at', 'cdc_op']
df_cdc = spark.createDataFrame(cdc_data, cdc_schema)

print('--- DỮ LIỆU CDC MỚI TỪ KAFKA ---')
df_cdc.show()

In [ ]:
# BƯỚC 3: THỰC THI LỆNH MERGE INTO BẰNG PYSPARK DELTA LAKE API
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, table_name)

# Thực hiện MERGE:
# 1. Khi trùng khóa và cờ là 'd' -> XÓA (Delete)
# 2. Khi trùng khóa và cờ là 'u' -> CẬP NHẬT (Update)
# 3. Khi chưa có khóa và cờ là 'i' -> CHÈN MỚI (Insert)
target_table.alias('tgt').merge(
    df_cdc.alias('src'),
    'tgt.customer_id = src.customer_id'
).whenMatchedDelete(
    condition = "src.cdc_op = 'd'"
).whenMatchedUpdate(
    condition = "src.cdc_op = 'u'",
    set = {
        'customer_name': 'src.customer_name',
        'customer_type': 'src.customer_type',
        'address': 'src.address',
        'updated_at': 'src.updated_at'
    }
).whenNotMatchedInsert(
    condition = "src.cdc_op = 'i'",
    values = {
        'customer_id': 'src.customer_id',
        'customer_name': 'src.customer_name',
        'customer_type': 'src.customer_type',
        'address': 'src.address',
        'updated_at': 'src.updated_at'
    }
).execute()

print('--- DỮ LIỆU SAU KHI MERGE INTO THÀNH CÔNG ---')
spark.table(table_name).show()

In [ ]:
# BƯỚC 4: KIỂM CHỨNG NHẬT KÝ LỊCH SỬ GIAO DỊCH (TRANSACTION LOG)
# Bạn sẽ thấy Delta Lake tự động sinh ra Version 1 với operation là 'MERGE'
# và operationMetrics ghi rõ: số dòng updated, deleted, inserted!
history_df = spark.sql(f'DESCRIBE HISTORY {table_name}')
history_df.select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

In [ ]:
# BƯỚC 5: CHỨNG MINH DỮ LIỆU CŨ KHÔNG HỀ MẤT (TIME TRAVEL)
# Dù CUST_002 đã bị XÓA và CUST_001 đã bị SỬA, ta vẫn có thể xem lại chính xác Version 0 ban đầu:
print('--- XEM LẠI DỮ LIỆU TẠI VERSION 0 (TRƯỚC KHI MERGE) ---')
df_v0 = spark.read.format('delta').option('versionAsOf', 0).table(table_name)
df_v0.show()